# ZeroAlpha Colab Research Training

This notebook is meant to run inside the connected Google Colab runtime. It uses the production `zeroalpha.cli` training/backtest path, but runs a much larger lower-timeframe HPO matrix than is practical on the local laptop.

Recommended bootstrap: clone the pushed GitHub branch into Colab. Upload mode remains as a fallback, but the normal path should not depend on the Colab upload widget.

In [ ]:
# Bootstrap the repo into /content/ZeroAlpha.
# In the Colab terminal, /Users/ethan/... does not exist. The default path clones GitHub.
# If the repo is private, add a Colab secret named GITHUB_TOKEN with repo read access.
from pathlib import Path
from urllib.parse import urlparse
import os
import shutil
import subprocess
import sys
import zipfile

BOOTSTRAP_MODE = "git"  # "git", "upload", or "existing"
GIT_URL = os.environ.get("ZEROALPHA_GIT_URL", "https://github.com/ea28/ZeroAlpha.git")
GIT_BRANCH = os.environ.get("ZEROALPHA_GIT_BRANCH", "codex/colab-research-training")
PROJECT_ROOT = Path("/content/ZeroAlpha") if Path("/content").exists() else Path.cwd()

def find_repo_root(root: Path) -> Path | None:
    for candidate in [root, *root.glob("*")]:
        if (candidate / "src" / "zeroalpha").exists():
            return candidate
    return None

def colab_secret(name: str) -> str:
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

def authenticated_git_url(url: str) -> str:
    token = colab_secret("GITHUB_TOKEN") or colab_secret("GH_TOKEN")
    parsed = urlparse(url)
    if token and parsed.scheme == "https" and parsed.netloc == "github.com":
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url

def safe_git_url(url: str) -> str:
    parsed = urlparse(url)
    if parsed.username or parsed.password:
        return url.replace(f"{parsed.username}:{parsed.password}@", "***@")
    return url

def clone_repo() -> None:
    clone_url = authenticated_git_url(GIT_URL)
    command = ["git", "clone", "--branch", GIT_BRANCH, "--single-branch", clone_url, str(PROJECT_ROOT)]
    print("Running:", " ".join(command[:-2] + [safe_git_url(clone_url), str(PROJECT_ROOT)]))
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode == 0:
        return
    hint = (
        "Git clone failed. If this repository is private, create a GitHub fine-grained token "
        "with read access to ea28/ZeroAlpha, add it in Colab Secrets as GITHUB_TOKEN, "
        "then rerun this cell. You can also set BOOTSTRAP_MODE='upload' as a fallback."
    )
    raise RuntimeError(f"{hint}\nexit_code={completed.returncode}")

if not (PROJECT_ROOT / "src" / "zeroalpha").exists():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if BOOTSTRAP_MODE == "git":
        if PROJECT_ROOT.exists():
            shutil.rmtree(PROJECT_ROOT)
        clone_repo()
    elif BOOTSTRAP_MODE == "upload":
        try:
            from google.colab import files
        except Exception as exc:
            raise RuntimeError("upload bootstrap requires a Colab notebook runtime") from exc
        print("Upload zeroalpha_colab_bundle.zip from the local repo root.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.endswith(".zip")]
        if not zip_names:
            raise RuntimeError("no .zip file uploaded")
        upload_path = Path(zip_names[0])
        if PROJECT_ROOT.exists():
            shutil.rmtree(PROJECT_ROOT)
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(upload_path) as archive:
            archive.extractall(PROJECT_ROOT)
        discovered = find_repo_root(PROJECT_ROOT)
        if discovered is not None and discovered != PROJECT_ROOT:
            temp_root = PROJECT_ROOT.with_name("ZeroAlpha_extracted")
            if temp_root.exists():
                shutil.rmtree(temp_root)
            discovered.rename(temp_root)
            shutil.rmtree(PROJECT_ROOT)
            temp_root.rename(PROJECT_ROOT)
    elif BOOTSTRAP_MODE != "existing":
        raise ValueError("BOOTSTRAP_MODE must be upload, git, or existing")

if not (PROJECT_ROOT / "src" / "zeroalpha").exists():
    raise RuntimeError(f"ZeroAlpha repo not found at {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PROJECT_ROOT", PROJECT_ROOT)
print("GIT_BRANCH", GIT_BRANCH)
print("Python", sys.version)


In [ ]:
# Install research dependencies in the Colab runtime.
# The project uses PYTHONPATH=src, so we avoid pip -e . because Colab may not match pyproject's Python pin.
%pip install -q --upgrade numpy scikit-learn lightgbm xgboost catboost pandas pyarrow duckdb tqdm matplotlib seaborn

# Optional foundation models. They can be slow and are not required for the broad tree-model HPO sweep.
# %pip install -q --upgrade tabpfn tabicl


In [ ]:
# Runtime / accelerator diagnostics.
import json
import platform
import subprocess

print("platform", platform.platform())
print("cwd", Path.cwd())
try:
    import torch
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
except Exception as exc:
    print("torch unavailable", exc)
try:
    import torch_xla.core.xla_model as xm
    print("TPU device", xm.xla_device())
except Exception as exc:
    print("TPU note: current sklearn/GBDT model stack mostly uses CPU/RAM; TPU is available only for future torch-xla models.", exc)

smoke = subprocess.run(
    [sys.executable, "-m", "zeroalpha.cli", "model", "smoke", "--models", "logistic,histgb,extratrees,lightgbm,xgboost,catboost", "--timeout-seconds", "120"],
    text=True,
    capture_output=True,
)
print(smoke.stdout[-4000:])
if smoke.returncode:
    print(smoke.stderr[-4000:])
    raise SystemExit(smoke.returncode)


In [ ]:
# Optional: keep cache/artifacts on Google Drive so long runs survive runtime resets.
from datetime import datetime, timezone

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE and Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ZeroAlphaColab")
else:
    STORAGE_ROOT = PROJECT_ROOT

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
CACHE_DIR = STORAGE_ROOT / "data" / "raw" / "binance"
ARTIFACT_DIR = STORAGE_ROOT / "artifacts" / "colab" / RUN_ID
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("CACHE_DIR", CACHE_DIR)
print("ARTIFACT_DIR", ARTIFACT_DIR)


In [ ]:
# Build the full lower-timeframe HPO matrix.
from zeroalpha.models.colab_research import (
    DEFAULT_MODELS,
    FAST_MODELS,
    build_experiment_matrix,
    run_experiments,
    summarize_artifacts,
    write_summary_csv,
)

# FAST_MODELS is already broad: logistic, histgb, extratrees, lightgbm, xgboost.
# DEFAULT_MODELS also adds randomforest and catboost, increasing runtime substantially.
MODEL_STACK = FAST_MODELS
EXPERIMENTS = build_experiment_matrix(include_15m=True, include_5m=True, include_1m=True, models=MODEL_STACK)
print("experiments", len(EXPERIMENTS))
for experiment in EXPERIMENTS[:10]:
    print(experiment.name)


In [ ]:
# Pilot: run a representative 15m subset first to verify data, speed, and artifact writing.
PILOT = [
    experiment for experiment in EXPERIMENTS
    if experiment.interval == "15m"
    and "champion_types" in experiment.name
    and experiment.selection_score == "expected_utility"
][:6]
pilot_results = run_experiments(
    PILOT,
    artifact_dir=ARTIFACT_DIR,
    cache_dir=CACHE_DIR,
    python_executable=sys.executable,
    resume=True,
)
pilot_results[:5]


In [ ]:
# Full matrix: 15m 6y, 5m 4y, and 1m 2y pilots, all with fold-local HPO.
# This is intentionally long. Leave the runtime running and rerun this cell to resume completed artifacts.
RUN_FULL_MATRIX = True
if RUN_FULL_MATRIX:
    full_results = run_experiments(
        EXPERIMENTS,
        artifact_dir=ARTIFACT_DIR,
        cache_dir=CACHE_DIR,
        python_executable=sys.executable,
        resume=True,
        timeout_seconds=None,
    )
else:
    full_results = summarize_artifacts(ARTIFACT_DIR)

summary_csv = ARTIFACT_DIR / "summary.csv"
write_summary_csv(full_results, summary_csv)
print("wrote", summary_csv)
full_results[:20]


In [ ]:
# Rank and inspect champions.
from dataclasses import asdict
import pandas as pd

results = summarize_artifacts(ARTIFACT_DIR)
df = pd.DataFrame([asdict(r) for r in results])
display_cols = [
    "name", "trades", "trades_per_day", "raw_candidates_per_day", "net_pnl",
    "sharpe", "profit_factor", "max_drawdown", "hit_rate", "samples", "artifact",
]
if not df.empty:
    ranked = df.sort_values(["sharpe", "net_pnl", "trades"], ascending=[False, False, False])
    display(ranked[display_cols].head(25))
    print("positive sharpe", int((df["sharpe"] > 0).sum()), "of", len(df))
    print("tpd >= 1", int((df["trades_per_day"] >= 1).sum()))
    print("tpd >= 2", int((df["trades_per_day"] >= 2).sum()))
    print("tpd >= 4", int((df["trades_per_day"] >= 4).sum()))
else:
    print("no result artifacts yet")


In [ ]:
# Cost stress for the best strict champions: rerun a few winners with higher spread/slippage manually if needed.
# The main matrix uses 4 bps spread + 1 bps slippage + 2 bps safety margin. A production candidate should survive stress.
if not df.empty:
    strict = df[(df["status"] == "ok") & (df["sharpe"] > 0) & (df["net_pnl"] > 0)]
    display(strict.sort_values(["trades_per_day", "sharpe"], ascending=[False, False])[display_cols].head(15))


In [ ]:
# Optional IBKR note.
# Colab usually cannot reach a local TWS/Gateway on your Mac unless you expose it on a reachable host/VPN/tunnel.
# For now, Binance archive is the deep-history source and IBKR quotes remain best for local execution-cost calibration.
IBKR_HOST = os.environ.get("ZEROALPHA_IBKR_HOST", "")
if IBKR_HOST:
    print("IBKR_HOST configured:", IBKR_HOST)
else:
    print("IBKR not configured in Colab. Use local IBKR quote recording for spread/slippage calibration.")


In [ ]:
# Bundle artifacts for download.
archive_path = ARTIFACT_DIR.with_suffix(".tar.gz")
subprocess.run(["tar", "-czf", str(archive_path), "-C", str(ARTIFACT_DIR.parent), ARTIFACT_DIR.name], check=True)
print("artifact archive", archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as exc:
    print("download skipped", exc)
